# كشكول التجارب: الشرح الرياضي لخوارزمية MMR بالأرقام (المحاضرة 46)

أهلاً بيك في معمل المحاضرة 46!
في الفيديو ده، المحاضر وضح لنا مشكلة كبيرة جداً في البحث العادي:
لو عندك مستندين بيتكلموا عن نفس الفكرة بالظبط، البحث العادي هيجيبهم هما الاتنين، وده بيضيع مساحة السياق في الـ LLM على كلام مكرر!
خوارزمية **Maximal Marginal Relevance (MMR)** بتحل المشكلة دي بمعادلة رياضية بسيطة جداً.

تعال نمشي مع المسألة الحسابية خطوة بخطوة بنفس الأرقام اللي المحاضر شرحها على السبورة!

### 1. تجهيز المسألة والبيانات
المحاضر افترض السؤال ده وثلاثة مستندات في قاعدة البيانات:

In [1]:
# السؤال: كيف نستخدم لانج تشين في الـ RAG؟
# المستند 1 (D1): LangChain enables retrieval with FAISS
# المستند 2 (D2): LangChain can use Chroma and Pinecone too
# المستند 3 (D3): LangChain agents can call external tools

# تشابه المستندات مع السؤال (Cosine Similarity مع Query):
sim_query = {
    "D1": 0.95,  # أعلى تشابه مع السؤال
    "D2": 0.93,  # تشابه عالي جداً مع السؤال
    "D3": 0.80   # تشابه جيد
}

# التشابه البيني بين المستندات وبعضها:
sim_between_docs = {
    ("D1", "D2"): 0.90,  # تشابه ضخم 90% (مستند مكرر!)
    ("D1", "D3"): 0.30,  # تشابه 30% فقط (مستند متنوع ومكمل!)
}

# معامل التوازن lambda
lambda_val = 0.7  # 70% للصلة بالسؤال، و 30% للتنوع ومنع التكرار

print("البيانات جاهزة للحساب!")

البيانات جاهزة للحساب!


### 2. الخطوة الأولى: اختيار المستند الأول صاحب أعلى تشابه
في الخطوة الأولى، بناخد المستند اللي عنده أعلى درجة تشابه مع السؤال فوراً:

In [2]:
selected = ["D1"]
remaining = ["D2", "D3"]

print(f"المستند الفائز بالمركز الأول: {selected[0]} (لأن درجته {sim_query['D1']})")
print(f"المستندات المتبقية للمركز الثاني: {remaining}")

المستند الفائز بالمركز الأول: D1 (لأن درجته 0.95)
المستندات المتبقية للمركز الثاني: ['D2', 'D3']


### 3. الخطوة الثانية: تطبيق معادلة MMR لاختيار المستند الثاني
معادلة MMR بتقول:
$$\text{Score} = (\lambda \times \text{Sim Query}) - ((1 - \lambda) \times \text{Sim D1})$$
تعال نحسبها للمستندين ونشوف مين هيكسب:

In [3]:
# حساب المستند D2
score_d2 = (lambda_val * sim_query["D2"]) - ((1 - lambda_val) * sim_between_docs[("D1", "D2")])
print(f"حساب D2: ({lambda_val} * 0.93) - (0.3 * 0.90) = {lambda_val*0.93:.3f} - {0.3*0.90:.3f} = {score_d2:.3f}")

# حساب المستند D3
score_d3 = (lambda_val * sim_query["D3"]) - ((1 - lambda_val) * sim_between_docs[("D1", "D3")])
print(f"حساب D3: ({lambda_val} * 0.80) - (0.3 * 0.30) = {lambda_val*0.80:.3f} - {0.3*0.30:.3f} = {score_d3:.3f}")

print(f"\nدرجة MMR لـ D2 هي: {score_d2:.3f}")
print(f"درجة MMR لـ D3 هي: {score_d3:.3f}")

حساب D2: (0.7 * 0.93) - (0.3 * 0.90) = 0.651 - 0.270 = 0.381
حساب D3: (0.7 * 0.80) - (0.3 * 0.30) = 0.560 - 0.090 = 0.470

درجة MMR لـ D2 هي: 0.381
درجة MMR لـ D3 هي: 0.470


### 4. النتيجة والترتيب النهائي
شايف النتيجة؟
رغم إن D2 كان أقرب للسؤال من D3 (0.93 مقابل 0.80)، إلا إن الخوارزمية عاقبته لأنه مكرر بنسبة 90% مع D1!

In [4]:
if score_d3 > score_d2:
    selected.append("D3")
else:
    selected.append("D2")

print("الترتيب النهائي للمستندات المسترجعة:")
print(f"المركز الأول: {selected[0]} -> أعلى صلة بالسؤال")
print(f"المركز الثاني: {selected[1]} -> أفضل توازن بين الصلة والتنوع!")

الترتيب النهائي للمستندات المسترجعة:
المركز الأول: D1 -> أعلى صلة بالسؤال
المركز الثاني: D3 -> أفضل توازن بين الصلة والتنوع!


### 5. جرب بنفسك!
طب لو خلينا lambda = 1.0 (يعني لغينا عقوبة التكرار تماماً)؟
جرب تغير قيمة lambda_val فوق وخليها 0.95 أو 1.0 وشوف هل D2 هيرجع يتصدر تاني ولا لأ!